# 02 – Validation setup & baseline

We fix **one** split that every model uses, so their AUC scores are directly comparable:

- **80 % development set**, split again into **5 stratified CV folds**. Used for tuning and for out-of-fold features such as target encoding.
- **20 % hold-out validation set**. Touched only to compare the final models and the ensemble.

The split is saved to `results/splits.csv` (`id`, `is_valid`, `fold`), so every later notebook loads exactly the same partition.
At the end we compute the **baseline every model has to beat**: the best single feature, scored on the validation set.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 42  # fixed random seed -> the split is identical every time the notebook runs

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

## Which kind of split mimics the real test set?
A validation set is only useful if it relates to the training data **the same way the Kaggle test set does**.
In the EDA, artists and songs repeated between train and test. So we measure how much test overlaps with train,
and compare that with a random split. If a random split gives the same overlap, it is a faithful simulation.
If test overlapped much less, we would need a split grouped by artist.

In [2]:
# a 'song' is identified by its title + artist (the same song can appear on several albums)
def song_key(df):
    return df["track_name"].astype(str) + "||" + df["artist_name"].astype(str)

# random 80/20 split; stratify keeps the 10 % hit rate equal in both parts
dev, valid = train_test_split(train, test_size=0.2, stratify=train["music_hit"], random_state=SEED)

# share of rows whose artist / song already appears in the 'training side'
overlap = pd.DataFrame({
    "Kaggle test vs. train":       [test["artist_name"].isin(set(train["artist_name"])).mean(),
                                    song_key(test).isin(set(song_key(train))).mean()],
    "random valid vs. dev":        [valid["artist_name"].isin(set(dev["artist_name"])).mean(),
                                    song_key(valid).isin(set(song_key(dev))).mean()],
}, index=["artist already seen", "same song (track+artist) already seen"]).round(3)
overlap

,Kaggle test vs. train,random valid vs. dev
artist already seen,0.957,0.954
same song (track+artist) already seen,0.161,0.140


In [3]:
# is the test set a later time period, or drawn from the same years?
year = lambda d: d["release_date"].str[:4].astype(int)
print("ids:   train", train["id"].min(), "-", train["id"].max(), "| test", test["id"].min(), "-", test["id"].max())
print("years: train mean %.2f median %d | test mean %.2f median %d"
      % (year(train).mean(), year(train).median(), year(test).mean(), year(test).median()))

ids:   train 1 - 350051 | test 350052 - 437564
years: train mean 2013.48 median 2017 | test mean 2013.47 median 2017


**Interpretation.** The random split reproduces the train/test relationship almost exactly
(about 95 % of artists already seen, about 15 % repeated songs), and test has the same year distribution as train.
So the Kaggle test set is a **random sample from the same pool**, not a later time period or a set of new artists.
A random stratified split is therefore the right choice. A split grouped by artist would underestimate our real score.

## Create and save the split

In [4]:
# one row per training song: is it in the validation set, and which CV fold is it in?
splits = pd.DataFrame({"id": train["id"], "is_valid": train["id"].isin(valid["id"]).astype(int), "fold": -1})

# divide the 80 % dev part into 5 folds with the same hit rate (validation rows keep fold = -1)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
dev_idx = splits.index[splits["is_valid"] == 0]
for k, (_, fold_idx) in enumerate(skf.split(dev_idx, train.loc[dev_idx, "music_hit"])):
    splits.loc[dev_idx[fold_idx], "fold"] = k

os.makedirs("results", exist_ok=True)  # all outputs go into results/
splits.to_csv("results/splits.csv", index=False)
splits.head()

,id,is_valid,fold
0,1,0,4
1,2,0,1
2,3,0,0
3,4,0,4
4,5,0,3


## Sanity check: same hit rate everywhere

In [5]:
# attach the split info to the labels and give every part a readable name
check = train[["music_hit"]].join(splits[["is_valid", "fold"]])
check["part"] = check["is_valid"].map({0: "dev", 1: "valid"}) + check["fold"].map(lambda f: f" fold {f}" if f >= 0 else "")

# number of rows and hit rate per part -> should all be ~10.34 %
check.groupby("part")["music_hit"].agg(rows="size", hit_rate="mean").round(4)

,rows,hit_rate
part,,
dev fold 0,56008,0.1034
dev fold 1,56008,0.1034
dev fold 2,56008,0.1034
dev fold 3,56008,0.1034
dev fold 4,56008,0.1034
valid,70011,0.1034


Every part has a hit rate of about 10.34 %, so the stratification worked.

## Baseline to beat: single features on the validation set
In the EDA we scored single features on all of train. To be fair to the models later, the baseline must be measured
**the same way the models will be measured**: on the 20 % validation set.

A feature can work in either direction (more followers = more hits, but more instrumental = *fewer* hits).
We decide the direction on the **dev** set only, then score that fixed direction on **valid**,
just as a model learns on dev and is evaluated on valid.

In [6]:
# the same simple features as in the EDA
def simple_features(df):
    out = df.select_dtypes("number").drop(columns=["id", "music_hit", "is_valid", "fold"], errors="ignore").copy()
    out["year"] = df["release_date"].str[:4].astype(int)
    out["genres_missing"] = df["artist_genres"].isna().astype(int)
    out["n_genres"] = df["artist_genres"].str.split("|").str.len().fillna(0)
    out["explicit"] = df["explicit_lyrics"].astype(int)
    return out

# reload dev/valid from the saved split, exactly like later notebooks will
data = train.merge(splits, on="id")
dev, valid = data[data["is_valid"] == 0], data[data["is_valid"] == 1]
X_dev, X_valid = simple_features(dev), simple_features(valid)

rows = []
for col in X_dev.columns:
    # +1 if higher values mean more hits on the dev set, -1 if the opposite
    direction = 1 if roc_auc_score(dev["music_hit"], X_dev[col]) >= 0.5 else -1
    # score the feature with that fixed direction on the validation set
    rows.append({"feature": col, "direction": direction,
                 "valid_auc": roc_auc_score(valid["music_hit"], direction * X_valid[col])})

baseline = pd.DataFrame(rows).sort_values("valid_auc", ascending=False).reset_index(drop=True)
baseline.to_csv("results/baseline_scores.csv", index=False)  # later notebooks compare against this file
baseline.round(4)

,feature,direction,valid_auc
0,artist_followers,1,0.8747
1,n_genres,1,0.7666
2,genres_missing,-1,0.6893
3,audio_loudness,1,0.6730
4,audio_instrumental,-1,0.6416
5,audio_energy,1,0.5982
6,audio_acoustic,-1,0.5608
7,explicit,1,0.5563
8,duration_sec,1,0.5281
9,year,-1,0.5259


In [7]:
# the single number every model has to beat
best = baseline.iloc[0]
print(f"BASELINE TO BEAT: '{best.feature}' alone -> validation AUC = {best.valid_auc:.4f}")

BASELINE TO BEAT: 'artist_followers' alone -> validation AUC = 0.8747


**How later notebooks use the split and the baseline:**
```python
splits = pd.read_csv("results/splits.csv")
train = train.merge(splits, on="id")
dev, valid = train[train.is_valid == 0], train[train.is_valid == 1]

baseline_auc = pd.read_csv("results/baseline_scores.csv")["valid_auc"].max()
```